# CBraMod and EEGSimpleConv on SHU-MI

# Part 1 — Method understanding and critical reading

![CBraMod architecture](figures/model.png)

![CBraMod architecture](figures/attention.png)

## 1. What are the claims made by the authors?

1. EEG plays an important role in brain–computer interfaces (BCIs) and healthcare applications, supporting tasks such as emotion recognition, motor imagery classification, and seizure detection.

2. The authors identify two main limitations of existing EEG foundation models: (1) many use full attention across all EEG patches to jointly model spatial and temporal dependencies, without explicitly distinguishing their different characteristics; and (2) positional encodings tied to fixed electrode identities may not adequately account for variations in channel configurations and referencing schemes, limiting adaptability across tasks and datasets.

3. The authors propose CBraMod, an EEG foundation model that divides each channel’s signal into equal-length, non-overlapping patches. Each patch is encoded through a convolutional time-domain branch and an FFT-based frequency-domain branch, whose outputs are added together. Asymmetric conditional positional encoding (ACPE) then incorporates information from a wider spatial neighborhood and a narrower temporal neighborhood. The resulting embeddings pass through stacked criss-cross transformer blocks. Within each block, half of the attention heads perform spatial attention across channels at a fixed time interval, while the other half perform temporal attention across time intervals within the same channel. The attention outputs are concatenated, and a feed-forward network further mixes their features. During pretraining, the model learns to reconstruct masked EEG patches using mean squared error. During downstream fine-tuning, the reconstruction head is replaced with a task-specific prediction head, and the backbone and prediction head are trained jointly.

## 2. What is the novelty?

The two main contributions are as follows:

1. Criss-cross attention: Parallel spatial and temporal attention mechanisms explicitly distinguish the two types of dependencies in EEG signals. Spatial attention captures relationships across channels within the same time interval, while temporal attention captures relationships across time intervals within the same channel.

2. Asymmetric conditional positional encoding (ACPE): A depthwise convolution with an asymmetric kernel ($k_s$ > $k_t$) generates positional embeddings from a wider spatial neighborhood and a narrower temporal neighborhood.

## 3. Are the ablations they run sufficient to support the claims?

Yes.

1. Fig 4: Comparisons with full attention, axial attention, and CCNet-style criss-cross attention across four datasets support the effectiveness of the proposed attention mechanism.

2. Fig 5:  Comparisons with no positional encoding, absolute positional encoding, and conventional conditional positional encoding across four datasets support the proposed asymmetric design.

3. Tab 4: Comparisons with no-pretraining and pretraining on uncleaned data support the benefits of both pretraining and data cleaning.

4. Appendix G: The author shows model architecture contributes more to the performance improvement compared to the larger size of pretraining dataset.

5. Appendix H: Longer pretraining segments generally improve performance, although the gains are relatively small.

6. Appendix I: Combining both branches (time-domain & frequency domain) generally outperforms either branch alone, supporting their complementary value.

7. Appendix J: Updating the entire model outperforms training only the classification head.

8. Appendix K: Low-resource performance; Appendix L: Ratios between spatial and temporal head; Appendix M: Masking ratios; Appendix N: Mask-token choices.



## 4. Tell us as well where you think the method is weak, and which questions you were left with after reading it.

weekness & problems:

1. Although the authors present it as a foundation model, it still requires full-parameter fine-tuning to achieve its best performance. It remains unclear whether a better-designed classification head could achieve comparable performance while keeping the pretrained encoder frozen.

2. ACPE depends on channel ordering, but robustness to reordered channels remains unclear. I would like to investigate how changing the order of EEG channels affects the model’s performance.



## Part 2: Reproducibility

In [5]:
import json
import sys
import subprocess
import time
from copy import deepcopy
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from IPython.display import display, Markdown

PROJECT = Path("/Users/leizan/PycharmProjects/SigmaNova")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from run_config import load_config, resolve_path
from preprocessing.shu_dataset import LoadDataset
from models.CBraMod.model_for_shu import Model as CBraMod
from models.EEGSimpleConv.model_for_shu_simpleconv import Model as EEGSimpleConv

METRICS = ["balanced_accuracy", "pr_auc", "auroc"]

# Configuration for the main experiment.
BASE_CONFIG_PATH = PROJECT / "configs/CBraMod.json"  #  "configs/CBraMod.json" or "configs/EEGSimpleConv.json"
base_cfg = load_config(BASE_CONFIG_PATH)

print("Selected model:", base_cfg["model"])


Selected model: cbramod


In [3]:
def make_loaders(config, batch_size=None, train_fraction=None, seed=None):
    """Create loaders using a config, with optional overrides."""
    params = SimpleNamespace(
        datasets_dir=str(resolve_path(PROJECT, config["datasets_dir"])),
        batch_size=(
            config["training"]["batch_size"]
            if batch_size is None else batch_size
        ),
        train_fraction=(
            config.get("train_fraction", 1.0)
            if train_fraction is None else train_fraction
        ),
        seed=config["seeds"][0] if seed is None else seed,
    )
    return LoadDataset(params).get_data_loader()


def run_experiment(config_path):
    """Launch the existing training script through its config runner."""
    subprocess.run(
        [
            sys.executable,
            str(PROJECT / "run_config.py"),
            "--config", str(config_path),
        ],
        cwd=PROJECT,
        check=True,
    )


def summarize_results(config):
    """Check saved settings and return formatted metrics for all seeds."""
    output_dir = resolve_path(PROJECT, config["output_dir"])
    paths = [
        output_dir / f"seed_{seed}" / "summary.json"
        for seed in config["seeds"]
    ]

    # Do not report a complete comparison from only some of the seeds.
    if not all(path.is_file() for path in paths):
        return None

    results = []
    for seed, path in zip(config["seeds"], paths):
        result = json.loads(path.read_text())
        saved = json.loads((path.parent / "config.json").read_text())

        assert saved["seed"] == seed
        assert saved.get("train_fraction", 1.0) == config.get(
            "train_fraction", 1.0
        )

        # Older CBraMod runs did not save a model-name field.
        if "model" in saved:
            assert saved["model"] == config["model"]

        expected = {
            **config["training"],
            **config["model_parameters"],
        }
        for key, value in expected.items():
            if key == "device":
                if value == "auto":
                    continue
                if value == "cuda":
                    value = f"cuda:{saved['cuda']}"

            if (
                key == "foundation_dir"
                and config["model_parameters"].get("use_pretrained_weights")
            ):
                value = str(resolve_path(PROJECT, value).resolve())

            assert saved[key] == value, f"Mismatched setting: {key}"

        results.append(result)

    statistics = {}
    for metric in METRICS:
        scores = [result[metric] for result in results]
        if len(scores) > 1:
            statistics[metric] = (
                f"{np.mean(scores):.4f} ± {np.std(scores, ddof=1):.4f}"
            )
        else:
            statistics[metric] = f"{scores[0]:.4f} (one seed)"

    return statistics

In [6]:
training_loaders = make_loaders(base_cfg)

signals, labels = next(iter(training_loaders["train"]))
print(signals.shape, labels.shape)
print(json.dumps(base_cfg, indent=2))

7210 2431 2347
11988
torch.Size([64, 32, 4, 200]) torch.Size([64])
{
  "model": "cbramod",
  "datasets_dir": "data/preprocessed",
  "seeds": [
    3407,
    3408,
    3409,
    3410,
    3411
  ],
  "output_dir": "models/CBraMod/original_full_01",
  "training": {
    "epochs": 2,
    "batch_size": 64,
    "lr": 0.0001,
    "weight_decay": 0.05,
    "optimizer": "AdamW",
    "clip_value": 1,
    "multi_lr": false,
    "device": "auto"
  },
  "model_parameters": {
    "use_pretrained_weights": true,
    "foundation_dir": "models/model_weights/CBraMod/pretrained_weights.pth",
    "classifier": "all_patch_reps",
    "dropout": 0.1,
    "frozen": false
  }
}


In [7]:
# fine-tune the model
run_experiment(BASE_CONFIG_PATH)

Namespace(model='cbramod', simpleconv_fm=128, simpleconv_n_convs=4, simpleconv_kernel_size=8, seed=3407, cuda=0, epochs=2, batch_size=64, lr=0.0001, weight_decay=0.05, optimizer='AdamW', clip_value=1.0, dropout=0.1, classifier='all_patch_reps', downstream_dataset='SHU-MI', datasets_dir='/Users/leizan/PycharmProjects/SigmaNova/data/preprocessed', train_fraction=1.0, train_keys=None, num_of_classes=2, model_dir='/Users/leizan/PycharmProjects/SigmaNova/models/CBraMod/original_full_01/seed_3407', num_workers=0, label_smoothing=0.1, multi_lr=False, frozen=False, use_pretrained_weights=True, foundation_dir='/Users/leizan/PycharmProjects/SigmaNova/models/model_weights/CBraMod/pretrained_weights.pth', device='cpu', check_only=False)
7210 2431 2347
11988
Model(
  (backbone): CBraMod(
    (patch_embedding): PatchEmbedding(
      (positional_encoding): Sequential(
        (0): Conv2d(200, 200, kernel_size=(19, 7), stride=(1, 1), padding=(9, 3), groups=200)
      )
      (proj_in): Sequential(
   

  0%|          | 0/113 [00:00<?, ?it/s]

Epoch 1 : Training Loss: 0.68650, acc: 0.64317, pr_auc: 0.72976, roc_auc: 0.71544, LR: 0.00005, Time elapsed 5.03 mins
[[701 511]
 [356 863]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.64317, pr_auc: 0.72976, roc_auc: 0.71544


  0%|          | 0/37 [00:00<?, ?it/s]

Epoch 2 : Training Loss: 0.65011, acc: 0.64917, pr_auc: 0.73435, roc_auc: 0.71775, LR: 0.00000, Time elapsed 4.93 mins
[[809 403]
 [450 769]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.64917, pr_auc: 0.73435, roc_auc: 0.71775
***************************Test************************


...

Epoch 1 : Training Loss: 0.68578, acc: 0.64327, pr_auc: 0.72916, roc_auc: 0.72201, LR: 0.00005, Time elapsed 4.84 mins
[[743 469]
 [398 821]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.64327, pr_auc: 0.72916, roc_auc: 0.72201


  0%|          | 0/37 [00:00<?, ?it/s]

Epoch 2 : Training Loss: 0.64858, acc: 0.64631, pr_auc: 0.75008, roc_auc: 0.72592, LR: 0.00000, Time elapsed 4.80 mins
[[813 399]
 [461 758]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.64631, pr_auc: 0.75008, roc_auc: 0.72592
***************************Test************************


100%|██████████| 37/37 [00:24<00:00,  1.50it/s]


***************************Test results************************
Test Evaluation: acc: 0.61917, pr_auc: 0.68077, roc_auc: 0.68883
[[749 420]
 [474 704]]
model save in /Users/leizan/PycharmProjects/SigmaNova/models/CBraMod/original_full_01/seed_3411/epoch2_acc_0.61917_pr_0.68077_roc_0.68883.pth
{
  "best_epoch": 2,
  "best_val_auroc": 0.7259247151130208,
  "balanced_accuracy": 0.6191708264286369,
  "pr_auc": 0.6807735874884593,
  "auroc": 0.6888340708832152,
  "confusion_matrix": [
    [
      749,
      420
    ],
    [
      474,
      704
    ]
  ],
  "checkpoint": "/Users/leizan/PycharmProjects/SigmaNova/models/CBraMod/original_full_01/seed_3411/epoch2_acc_0.61917_pr_0.68077_roc_0.68883.pth",
  "model": "cbramod",
  "seed": 3411,
  "epochs": 2,
  "scope": "pilot",
  "multi_lr": false,
  "device": "cpu",
  "pretrained": true,
  "frozen": false,
  "python": "3.12.6",
  "torch": "2.14.0",
  "train_keys": null,
  "train_fraction": 1.0,
  "split_sizes": {
    "train": 7210,
    "val": 243

In [8]:
statistics = summarize_results(base_cfg)

if statistics is None:
    print("Experiment incomplete.")
else:
    print(
        f"{base_cfg['model']}: "
        f"{base_cfg['training']['epochs']} epochs × "
        f"{len(base_cfg['seeds'])} seeds"
    )
    for metric, value in statistics.items():
        print(f"{metric}: {value}")

cbramod: 2 epochs × 5 seeds
balanced_accuracy: 0.6255 ± 0.0052
pr_auc: 0.6886 ± 0.0104
auroc: 0.6947 ± 0.0058


The final results:

| Model                    | Balanced accuracy ↑ | AUC-PR ↑ | AUROC ↑ |
|--------------------------|--------------------:|---:|---:|
| EEGSimpleConv (2 epochs) |     0.5001 ± 0.0005 | 0.5021 ± 0.0057 | 0.5016 ± 0.0062 |
| CBraMod (2 epochs)       |     0.6255 ± 0.0052 | 0.6886 ± 0.0104 | 0.6947 ± 0.0058 |
| CBraMod (Published)      |     0.6370 ± 0.0151 | 0.7139 ± 0.0088 | 0.6988 ± 0.0068 |

## Part 3: Comparison

### （1） Classification performance

| Model                    | Balanced accuracy ↑ | AUC-PR ↑ | AUROC ↑ |
|--------------------------|--------------------:|---:|---:|
| EEGSimpleConv (2 epochs) |     0.5001 ± 0.0005 | 0.5021 ± 0.0057 | 0.5016 ± 0.0062 |
| CBraMod (2 epochs)       |     0.6255 ± 0.0052 | 0.6886 ± 0.0104 | 0.6947 ± 0.0058 |
| CBraMod (Published)      |     0.6370 ± 0.0151 | 0.7139 ± 0.0088 | 0.6988 ± 0.0068 |

### （2） Data efficiency

In [ ]:
MODEL_NAMES = ["CBraMod", "EEGSimpleConv"]
FRACTIONS = [0.25, 0.50]
EFFICIENCY_SEEDS = [3407, 3408, 3409, 3410, 3411]
EFFICIENCY_EPOCHS = 2
EFFICIENCY_ROOT = PROJECT / "results" / "data_efficiency"

base_configs = {
    name: load_config(PROJECT / "configs" / f"{name}.json")
    for name in MODEL_NAMES
}


config_dir = EFFICIENCY_ROOT / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

efficiency_config_paths = []

for model_name in MODEL_NAMES:
    for fraction in FRACTIONS:
        experiment_cfg = deepcopy(base_configs[model_name])
        experiment_cfg["train_fraction"] = fraction
        experiment_cfg["seeds"] = EFFICIENCY_SEEDS
        experiment_cfg["training"]["epochs"] = EFFICIENCY_EPOCHS
        experiment_cfg["output_dir"] = str(
            EFFICIENCY_ROOT / model_name / f"pct_{int(fraction * 100)}"
        )

        config_path = config_dir / f"{model_name}_{int(fraction * 100)}.json"

        config_path.write_text(json.dumps(experiment_cfg, indent=2))

        efficiency_config_paths.append(config_path)

for config_path in efficiency_config_paths:
        experiment_cfg = load_config(config_path)

        if summarize_results(experiment_cfg) is not None:
            print("Already complete:", config_path.name)
            continue

        run_experiment(config_path)

Namespace(model='cbramod', simpleconv_fm=128, simpleconv_n_convs=4, simpleconv_kernel_size=8, seed=3407, cuda=0, epochs=2, batch_size=64, lr=0.0001, weight_decay=0.05, optimizer='AdamW', clip_value=1.0, dropout=0.1, classifier='all_patch_reps', downstream_dataset='SHU-MI', datasets_dir='/Users/leizan/PycharmProjects/SigmaNova/data/preprocessed', train_fraction=0.25, train_keys=None, num_of_classes=2, model_dir='/Users/leizan/PycharmProjects/SigmaNova/results/data_efficiency/CBraMod/pct_25/seed_3407', num_workers=0, label_smoothing=0.1, multi_lr=False, frozen=False, use_pretrained_weights=True, foundation_dir='/Users/leizan/PycharmProjects/SigmaNova/models/model_weights/CBraMod/pretrained_weights.pth', device='cpu', check_only=False)
1802 2431 2347
6580
Model(
  (backbone): CBraMod(
    (patch_embedding): PatchEmbedding(
      (positional_encoding): Sequential(
        (0): Conv2d(200, 200, kernel_size=(19, 7), stride=(1, 1), padding=(9, 3), groups=200)
      )
      (proj_in): Sequenti

  0%|          | 0/29 [00:00<?, ?it/s]

Epoch 1 : Training Loss: 0.69982, acc: 0.50041, pr_auc: 0.71131, roc_auc: 0.68437, LR: 0.00005, Time elapsed 1.56 mins
[[1212    0]
 [1218    1]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.50041, pr_auc: 0.71131, roc_auc: 0.68437


  0%|          | 0/37 [00:00<?, ?it/s]

Epoch 2 : Training Loss: 0.68634, acc: 0.60127, pr_auc: 0.72574, roc_auc: 0.69387, LR: 0.00000, Time elapsed 1.62 mins
[[503 709]
 [259 960]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.60127, pr_auc: 0.72574, roc_auc: 0.69387
***************************Test************************


...

Epoch 2 : Training Loss: 0.68847, acc: 0.55372, pr_auc: 0.63261, roc_auc: 0.66001, LR: 0.00000, Time elapsed 1.45 mins
[[ 166 1046]
 [  36 1183]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.55372, pr_auc: 0.63261, roc_auc: 0.66001
***************************Test************************


100%|██████████| 37/37 [00:23<00:00,  1.55it/s]


***************************Test results************************
Test Evaluation: acc: 0.50557, pr_auc: 0.58692, roc_auc: 0.60337
[[  15 1154]
 [   2 1176]]
model save in /Users/leizan/PycharmProjects/SigmaNova/results/data_efficiency/CBraMod/pct_25/seed_3410/epoch2_acc_0.50557_pr_0.58692_roc_0.60337.pth
{
  "best_epoch": 2,
  "best_val_auroc": 0.6600142274276648,
  "balanced_accuracy": 0.5055668435140391,
  "pr_auc": 0.5869223652997493,
  "auroc": 0.6033667566637281,
  "confusion_matrix": [
    [
      15,
      1154
    ],
    [
      2,
      1176
    ]
  ],
  "checkpoint": "/Users/leizan/PycharmProjects/SigmaNova/results/data_efficiency/CBraMod/pct_25/seed_3410/epoch2_acc_0.50557_pr_0.58692_roc_0.60337.pth",
  "model": "cbramod",
  "seed": 3410,
  "epochs": 2,
  "scope": "pilot",
  "multi_lr": false,
  "device": "cpu",
  "pretrained": true,
  "frozen": false,
  "python": "3.12.6",
  "torch": "2.14.0",
  "train_keys": null,
  "train_fraction": 0.25,
  "split_sizes": {
    "train": 18

 16%|█▌        | 6/38 [00:07<00:40,  1.27s/it]

Epoch 1 : Training Loss: 0.69794, acc: 0.50000, pr_auc: 0.59913, roc_auc: 0.60934, LR: 0.00005, Time elapsed 2.36 mins
[[1212    0]
 [1219    0]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.50000, pr_auc: 0.59913, roc_auc: 0.60934
Epoch 2 : Training Loss: 0.69398, acc: 0.60355, pr_auc: 0.65954, roc_auc: 0.65459, LR: 0.00000, Time elapsed 2.67 mins
[[943 269]
 [696 523]]
roc_auc increasing....saving weights !! 
Val Evaluation: acc: 0.60355, pr_auc: 0.65954, roc_auc: 0.65459
***************************Test************************
***************************Test results************************
Test Evaluation: acc: 0.54691, pr_auc: 0.57242, roc_auc: 0.58814
[[976 193]
 [873 305]]
model save in /Users/leizan/PycharmProjects/SigmaNova/results/data_efficiency/CBraMod/pct_25/seed_3411/epoch2_acc_0.54691_pr_0.57242_roc_0.58814.pth
{
  "best_epoch": 2,
  "best_val_auroc": 0.6545855364863804,
  "balanced_accuracy": 0.5469075189422271,
  "pr_auc": 0.5724230344366443,
  "auroc"

In [ ]:
table = [
    "| Model | Training data | Balanced accuracy | AUC-PR | AUROC |",
    "|---|---:|---:|---:|---:|",
]

for config_path in efficiency_config_paths:
    experiment_cfg = load_config(config_path)
    statistics = summarize_results(experiment_cfg)

    values = (
        [statistics[metric] for metric in METRICS]
        if statistics is not None
        else ["Pending"] * 3
    )

    table.append(
        f"| {experiment_cfg['model']} "
        f"| {experiment_cfg['train_fraction']:.0%} "
        f"| {' | '.join(values)} |"
    )

display(Markdown("\n".join(table)))

| Model | Training data | Balanced accuracy | AUC-PR | AUROC |
|---|--------------:|---:|---:|---:|
| CBraMod (2 epochs) |           25% | 0.5457 ± 0.0438 | 0.6139 ± 0.0326 | 0.6260 ± 0.0286 |
| CBraMod (2 epochs) |           50% | 0.6016 ± 0.0213 | 0.6658 ± 0.0206 | 0.6698 ± 0.0168 |
| CBraMod (2 epochs) |          100% |     0.6255 ± 0.0052 | 0.6886 ± 0.0104 | 0.6947 ± 0.0058 |
| EEGSimpleConv (2 epochs)  |           25% | 0.5005 ± 0.0007 | 0.4995 ± 0.0063 | 0.5030 ± 0.0073 |
| EEGSimpleConv (2 epochs)  |           50% | 0.5001 ± 0.0002 | 0.4981 ± 0.0145 | 0.4951 ± 0.0183 |
| EEGSimpleConv (2 epochs)  |          100% |     0.5001 ± 0.0005 | 0.5021 ± 0.0057 | 0.5016 ± 0.0062 |

### （3）Computational efficiency

In [4]:
def load_model(model_class, model_name):
    folder = (
        PROJECT / "models" / model_name
        / "original_full_01" / "seed_3407"
    )

    config = json.loads((folder / "config.json").read_text())
    summary = json.loads((folder / "summary.json").read_text())

    # Load the full fine-tuned checkpoint instead of the pretraining weights.
    config["use_pretrained_weights"] = False
    model = model_class(SimpleNamespace(**config))

    weights = torch.load(
        summary["checkpoint"],
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(weights)
    return model.eval()


def measure_inference_time(model, test_loader):
    model = model.cpu().eval()
    times_ms = []

    with torch.inference_mode():
        # Warm-up predictions are not timed.
        signals, _ = next(iter(test_loader))
        signals = signals.cpu()

        for _ in range(50):
            model(signals)

        for signals, _ in test_loader:
            signals = signals.cpu()

            start = time.perf_counter()
            model(signals)
            times_ms.append((time.perf_counter() - start) * 1000)

    return np.mean(times_ms), np.std(times_ms, ddof=1)

inference_loaders = make_loaders(
    base_cfg,
    batch_size=1,
    train_fraction=1.0,
)

inference_models = {
    "CBraMod": load_model(CBraMod, "CBraMod"),
    "EEGSimpleConv": load_model(EEGSimpleConv, "EEGSimpleConv"),
}

torch.set_num_threads(4)

for name, model in inference_models.items():
    mean_ms, std_ms = measure_inference_time(
        model,
        inference_loaders["test"],
    )
    print(f"{name}: {mean_ms:.2f} ± {std_ms:.2f} ms/sample")

7210 2431 2347
11988
CBraMod: 18.78 ± 0.63 ms/sample
EEGSimpleConv: 2.62 ± 0.17 ms/sample


The table below shows the mean ± standard deviation of inference time across individual samples in the SHU-MI test set, measured with batch size 1:

| Model | Inference time (ms/sample) ↓ |
|---|---:|
| CBraMod | 19.85 ± 7.91 |
| EEGSimpleConv | 2.18 ± 0.12 |


The table below shows the size of different models:
| Component |              CBraMod | EEGSimpleConv |
|---|---------------------:|--------------:|
| Encoder parameters |                   4M |          3.4M |
| Classification head parameters |                  20M |           360 |
| **Total parameters** |  **24M** |      **3.4M** |

Comments:

1. Based on our two-epoch experiments, I would prefer CBraMod for its classification performance and data efficiency. EEGSimpleConv may be preferable when model size and inference latency are priorities, provided its classification performance is acceptable.

2. In the data-efficiency experiments, CBraMod’s performance improves as the amount of labeled fine-tuning data increases. EEGSimpleConv shows no clear improvement under the same setting.

## Part 4: Proposing Improvements

Improvements:

**Potential improvements:**

1. **Simplify the classification head:** The current CBraMod classifier contains approximately 20M parameters, compared with 4M in the encoder. A smaller head may reduce overfitting on limited labeled data. I would start by testing `all_patch_reps_onelayer` and `avgpooling_patch_reps`. As an alternative, I would extract features from an intermediate layer of the trained classification head and use them to train a random forest classifier.

2. **Add mild EEG augmentation:** Introduce small amounts of additive noise and random channel dropout during training to improve robustness to imperfect recordings.

3. **Investigate batch-size effects:** Study how batch size affects convergence and performance.

4. **Explore contrastive pretraining:** Replace or complement the reconstruction loss with a contrastive objective to encourage consistent representations of different augmented views of the same EEG sample. I would evaluate whether these representations improve downstream classification.

5. **Encode temporal changes explicitly:** Add a branch that encodes first-order differences between consecutive signal values, complementing the existing time-domain and frequency-domain representations.

7. **Use causal attention mechanisme:** Use a causal foundation model to estimate potential causal relationships between EEG channels and incorporate this information into the attention mechanism.

6. **Investigate channel-order sensitivity:** Measure how predictions change when EEG channels are reordered. If the model is sensitive, explore positional encoding based on electrode coordinates rather than channel-list order.